# Exercício 1 — Engenharia de prompts: requisitos de conteúdo para aumentar a qualidade das respostas de um LLM

Este notebook segue a tarefa da **Aula 12** (Agentes, seus modelos e "IA" local):

> **Comentar sobre os requisitos de conteúdo de um prompt para aumentar a qualidade das respostas a uma consulta a um modelo de LLM.**

A abordagem é a mesma dos exercícios anteriores: analisamos os PDFs da própria aula
(via `pdftotext`), usamos esse material como **contexto** e pedimos ao **Ollama** que
gere a melhor resposta possível, fundamentada nesse conteúdo. Ao final, geramos um PDF
para entrega no Moodle.

## 1. Contexto

A qualidade da resposta de um LLM depende fortemente do **conteúdo do prompt**. A Aula 12
trata, no Tópico 12.03, dos *rudimentos sobre especificações de "prompts"* — como
caracterizar agentes, mecanismos, processos e a forma de apresentação dos resultados por
meio de tarefas bem especificadas.

A literatura de **engenharia de prompts** (ver referências da ementa) converge em alguns
requisitos de conteúdo recorrentes: definir **papel/persona**, **objetivo/tarefa**,
**contexto**, **restrições**, **formato de saída**, **exemplos (few-shot)**, **critérios
de qualidade** e a **decomposição da tarefa** em passos.

## 2. Célula auxiliar — imports e descoberta de diretórios

Execute a célula abaixo para configurar imports e os diretórios de saída.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import json
import glob
import shutil
import subprocess
from urllib.request import Request, urlopen

_repo = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_repo, "setup.sh")):
        break
    _repo = os.path.dirname(_repo)
PASTA_MOODLE = os.path.join(_repo, "work", "tarefas_moodle")
os.makedirs(PASTA_MOODLE, exist_ok=True)
PASTA_AULA = os.path.join(PASTA_MOODLE, "tarefa_aula12")
os.makedirs(PASTA_AULA, exist_ok=True)
os.environ["PASTA_AULA_SAVE"] = PASTA_AULA
print("Diretorio de saida:", PASTA_AULA)


Diretorio de saida: /l/disk0/viniciusd/Projetos/simserv_jupyter_env/work/tarefas_moodle/tarefa_aula12


## 3. Análise dos PDFs da aula

Extraímos o texto dos PDFs da Aula 12 com `pdftotext` (poppler). O texto extraído será o
**contexto** enviado ao LLM para fundamentar a resposta. O Tópico 12.03 (prompts) vem
primeiro por ser o mais relevante para a tarefa.

In [2]:
# Extrai o texto dos PDFs da Aula 12 (via pdftotext) para usar como contexto do LLM.
PDFS_AULA12 = [
    "260602_topico_12_03_consultas_linguagem_natural.pdf",  # prompts (mais relevante)
    "260602_topico_12_02_ollama_informacoes.pdf",
    "260602_topico12_01_Agentes_Modelos_de_251028_V3.pdf",
    "260602_Aula12_IA006_2026_ementa.pdf",
]
# Procura os PDFs em varias pastas candidatas (a do notebook e a de tarefas).
PASTAS_CANDIDATAS = [
    os.getcwd(),
    os.path.join(_repo, "tarefas", "aula12"),
    os.path.join(_repo, "tarefas"),
]
MAX_CHARS_POR_PDF = 5000  # limita o contexto para caber na janela do modelo


def localizar_pdf(nome):
    for pasta in PASTAS_CANDIDATAS:
        caminho = os.path.join(pasta, nome)
        if os.path.exists(caminho):
            return caminho
    return None


def extrair_texto_pdf(caminho, max_chars):
    try:
        out = subprocess.run(["pdftotext", "-layout", caminho, "-"],
                             capture_output=True, text=True, timeout=60)
        return out.stdout.strip()[:max_chars]
    except Exception as e:
        return f"[falha ao extrair {os.path.basename(caminho)}: {e}]"


partes = []
pdfs_analisados = []
for nome in PDFS_AULA12:
    caminho = localizar_pdf(nome)
    if caminho:
        texto = extrair_texto_pdf(caminho, MAX_CHARS_POR_PDF)
        partes.append(f"===== {nome} =====\n{texto}")
        pdfs_analisados.append(nome)
        print(f"OK  ({len(texto):>5} chars): {nome}")
    else:
        print(f"NAO encontrado em nenhuma pasta candidata: {nome}")

contexto_pdfs = "\n\n".join(partes)
print(f"\nContexto total: {len(contexto_pdfs)} chars de {len(pdfs_analisados)} PDF(s).")
if not pdfs_analisados:
    print("AVISO: nenhum PDF encontrado. Coloque os PDFs da Aula 12 em tarefas/aula12/.")


OK  ( 5000 chars): 260602_topico_12_03_consultas_linguagem_natural.pdf
OK  ( 5000 chars): 260602_topico_12_02_ollama_informacoes.pdf
OK  ( 5000 chars): 260602_topico12_01_Agentes_Modelos_de_251028_V3.pdf
OK  ( 5000 chars): 260602_Aula12_IA006_2026_ementa.pdf

Contexto total: 20237 chars de 4 PDF(s).


## 4. Pergunta

> **Comente sobre os requisitos de conteúdo de um prompt para aumentar a qualidade das
> respostas a uma consulta a um modelo de LLM.**

A resposta pode ser gerada pela IA (seção 5) e editada manualmente antes da exportação.

In [3]:
resposta_pergunta_1 = """
Escreva sua resposta aqui (ou gere com a IA na secao 5 e edite se desejar).
"""


## 5. Gerar a melhor resposta com IA (fundamentada nos PDFs)

A célula abaixo envia ao Ollama a tarefa **junto com o material extraído dos PDFs** e
guarda o resultado em `resposta_pergunta_1` (que vai para o PDF). Revise/edite se quiser.

In [4]:
MODELO = os.getenv("OLLAMA_MODEL", "ministral-3:3b")
BASE_URL = os.getenv("OLLAMA_BASE_URL", "https://api.ollama.com").rstrip("/")
API_KEY = os.getenv("OLLAMA_API_KEY")

prompt_resposta = f"""Voce e um especialista em engenharia de prompts de um curso de
ciencia e engenharia dos sistemas complexos. Responda em portugues, de forma clara,
organizada e objetiva (no maximo ~250 palavras).

TAREFA: Comentar sobre os requisitos de conteudo de um prompt para aumentar a qualidade
das respostas a uma consulta a um modelo de LLM.

Enumere e explique os principais requisitos de conteudo de um bom prompt (por exemplo:
papel/persona, objetivo e tarefa bem definidos, contexto e dados relevantes, restricoes e
limites, formato de saida desejado, exemplos (few-shot), criterios de qualidade e
decomposicao da tarefa em passos). Para cada requisito, explique brevemente por que ele
melhora a qualidade da resposta. Fundamente-se no material da Aula 12 transcrito abaixo
quando for pertinente.

MATERIAL DA AULA 12 (trechos):
{contexto_pdfs}
"""

data = json.dumps({
    "model": MODELO,
    "prompt": prompt_resposta,
    "stream": False,
    "options": {"temperature": 0.3, "top_p": 0.9, "num_predict": 800},
}).encode()

headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
}
if API_KEY:
    headers["Authorization"] = f"Bearer {API_KEY}"

req = Request(BASE_URL + "/api/generate", data=data, headers=headers)

try:
    resp = json.loads(urlopen(req, timeout=120).read().decode())
    resposta = resp["response"].strip()
    if resposta:
        # A resposta da IA vira a resposta_pergunta_1 que vai para o PDF (secao 6).
        resposta_pergunta_1 = resposta
        print("Resposta gerada com sucesso!")
    else:
        print("IA retornou resposta vazia. Mantida a resposta atual.")
except Exception as e:
    print(f"Erro ao consultar IA: {type(e).__name__}: {e}")

print()
print("Resposta atual (sera usada no PDF):")
print(resposta_pergunta_1)


Resposta gerada com sucesso!

Resposta atual (sera usada no PDF):
**Requisitos de conteúdo para prompts em engenharia de sistemas complexos**

Um prompt bem estruturado em ciência e engenharia dos sistemas complexos (CESC) deve seguir princípios que garantem **precisão, coerência e utilidade operacional**, especialmente quando interagindo com modelos LLM (como o Ollama no SimServ). Os principais requisitos são:

1. **Papel/persona (role specification)**
   *Exemplo*: *"Agente especialista em modelagem baseada em agentes para sistemas sociais, seguindo os paradigmas do Agent-Zero (Epstein)."*
   **Porque?** Define o *domínio de expertise* do LLM, evitando respostas genéricas ou irrelevantes para tarefas específicas (ex.: extrair variáveis de um modelo de epidemia).

2. **Objetivo e tarefa bem definidos**
   *Exemplo*: *"Identifique 3 variáveis globais do modelo de interação entre predadores e presas, listando suas unidades de medida e papel na dinâmica emergente."*
   **Porque?** Clarif

## 6. Demonstração: prompt simples vs. prompt estruturado

Para tornar concretos os "requisitos de conteúdo", fazemos a **mesma consulta** ao LLM de
duas formas: (a) apenas a pergunta crua e (b) um prompt estruturado com papel, tarefa,
contexto, restrições e formato. Compare a diferença de qualidade nas respostas.

In [5]:
def consultar_ollama(prompt, num_predict=400, temperature=0.3, timeout=120):
    data = json.dumps({
        "model": MODELO, "prompt": prompt, "stream": False,
        "options": {"temperature": temperature, "top_p": 0.9, "num_predict": num_predict},
    }).encode()
    headers = {"Content-Type": "application/json", "User-Agent": "Mozilla/5.0",
               "Accept": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    req = Request(BASE_URL + "/api/generate", data=data, headers=headers)
    try:
        resp = json.loads(urlopen(req, timeout=timeout).read().decode())
        return resp["response"].strip()
    except Exception as e:
        return f"[Erro ao consultar IA: {type(e).__name__}: {e}]"


# Mesma consulta, dois prompts: um "cru" e um estruturado com os requisitos de conteudo.
consulta = "Como modelar uma comunidade de agentes que cooperam e competem por um recurso comum?"

prompt_simples = consulta

prompt_estruturado = f"""[PAPEL] Voce e um especialista em modelagem baseada em agentes (ABM).
[TAREFA] {consulta}
[CONTEXTO] Curso de sistemas complexos; objetivo e simular a emergencia de cooperacao/competicao.
[RESTRICOES] Use o protocolo ODD (Overview, Design concepts, Details); cite entidades, variaveis,
parametros e regras de interacao; seja conciso (~150 palavras).
[FORMATO] Responda em topicos curtos: Agentes, Variaveis, Parametros, Regras, Saidas esperadas."""

print("Consultando com PROMPT SIMPLES...")
resp_simples = consultar_ollama(prompt_simples, num_predict=400)
print("Consultando com PROMPT ESTRUTURADO...")
resp_estruturado = consultar_ollama(prompt_estruturado, num_predict=400)

print("\n=== RESPOSTA AO PROMPT SIMPLES ===\n")
print(resp_simples[:700])
print("\n=== RESPOSTA AO PROMPT ESTRUTURADO ===\n")
print(resp_estruturado[:700])


Consultando com PROMPT SIMPLES...


Consultando com PROMPT ESTRUTURADO...



=== RESPOSTA AO PROMPT SIMPLES ===

Modelar uma **comunidade de agentes que cooperam e competem por um recurso comum** é um problema clássico em **teoria dos jogos, ciência da complexidade e sistemas adaptativos**. Essa abordagem pode ser aplicada em diversos contextos, como **ecologia, economia, política, robótica colaborativa e IA social**.

Aqui está uma estrutura detalhada para modelar tal sistema:

---

### **1. Definição dos Componentes Principais**
#### **A. Agentes**
- **Características**:
  - **Objetivos**: Cada agente tem um objetivo individual (ex: maximizar sua utilidade, sobrevivência, riqueza).
  - **Ações**: Decisões sobre como usar o recurso (ex: consumir, compartilhar, proteger).
  - **Conhecimento**: Informaç

=== RESPOSTA AO PROMPT ESTRUTURADO ===

### **Modelagem de Comunidade Cooperativa/Competitiva (ABM)**

#### **Agentes**
- **Entidades**:
  - *Agente Cooperativo*: Prioriza compartilhar o recurso (ex.: planta que poliniza).
  - *Agente Competitivo*: Consome o re

## 7. Exportar para o Moodle

Gera o Markdown e o PDF completos (material analisado + resposta + demonstração).

In [6]:
from datetime import datetime

NOME = os.getenv("NOME", "nao informado")
RA = os.getenv("RA", "nao informado")
MODELO = os.getenv("OLLAMA_MODEL", "nao informado")
agora = datetime.now().strftime("%Y-%m-%d %H:%M")


def _bloco_citacao(texto):
    return "> " + texto.strip().replace("\n", "\n> ")


PATH_MD = os.path.join(PASTA_AULA, "aula_12_exercicio_01.md")
with open(PATH_MD, "w", encoding="utf-8") as f:
    f.write("# Aula 12 — Requisitos de conteudo de um prompt para LLMs\n\n")
    f.write(f"_Aluno: {NOME} (RA: {RA})_\n\n")
    f.write(f"_Modelo LLM: {MODELO}_\n\n")
    f.write(f"_Executado em: {agora}_\n\n")
    f.write("---\n\n")
    f.write("## 1. Material analisado\n\n")
    f.write("Os PDFs da Aula 12 abaixo foram lidos (via `pdftotext`) e usados como "
            "contexto para fundamentar a resposta gerada pela IA:\n\n")
    for nome in pdfs_analisados:
        f.write(f"- `{nome}`\n")
    f.write("\n---\n\n")
    f.write("## 2. Resposta: requisitos de conteudo de um bom prompt\n\n")
    f.write(resposta_pergunta_1.strip() + "\n\n")
    if "resp_simples" in globals() and "resp_estruturado" in globals():
        f.write("---\n\n")
        f.write("## 3. Demonstracao: prompt simples vs. prompt estruturado\n\n")
        f.write(f"**Consulta:** {consulta}\n\n")
        f.write("**(a) Prompt simples (apenas a pergunta):**\n\n")
        f.write(_bloco_citacao(resp_simples) + "\n\n")
        f.write("**(b) Prompt estruturado (papel + tarefa + contexto + restricoes + formato):**\n\n")
        f.write(_bloco_citacao(resp_estruturado) + "\n\n")

PATH_PDF = PATH_MD.replace(".md", ".pdf")
if shutil.which("pandoc"):
    # -f markdown-yaml_metadata_block: a saida do LLM pode conter linhas "---" que o
    # pandoc interpretaria como bloco YAML de metadados (e falharia). Desabilitamos isso.
    subprocess.run(["pandoc", "-f", "markdown-yaml_metadata_block",
                   PATH_MD, "-o", PATH_PDF,
                   "--pdf-engine=xelatex", "--resource-path", PASTA_AULA,
                   "-V", "mainfont=DejaVu Serif"], check=True)
    print("PDF gerado.")
else:
    print("Aviso: pandoc nao instalado. Execute ./setup.sh para instalar.")
print(f"Markdown: {os.path.abspath(PATH_MD)}")
print(f"PDF:      {os.path.abspath(PATH_PDF)}")


PDF gerado.
Markdown: /l/disk0/viniciusd/Projetos/simserv_jupyter_env/work/tarefas_moodle/tarefa_aula12/aula_12_exercicio_01.md
PDF:      /l/disk0/viniciusd/Projetos/simserv_jupyter_env/work/tarefas_moodle/tarefa_aula12/aula_12_exercicio_01.pdf
